In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Check if QLoRA v2 saved any checkpoints
import os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')

# Check v2 checkpoints
v2_path = DRIVE / 'models' / 'qwen-qlora-v2' / 'checkpoints'
print("=== QLoRA v2 ===")
if v2_path.exists():
    for item in sorted(v2_path.iterdir()):
        if item.is_dir():
            files = list(item.iterdir())
            print(f"  {item.name}/ ({len(files)} files)")
        else:
            print(f"  {item.name}")
else:
    print("  NOT FOUND — no v2 checkpoints saved")

# Also check v1 (the broken one)
v1_path = DRIVE / 'models' / 'qwen-qlora' / 'checkpoints'
print("\n=== QLoRA v1 (broken) ===")
if v1_path.exists():
    for item in sorted(v1_path.iterdir()):
        if item.is_dir():
            print(f"  {item.name}/")
        else:
            print(f"  {item.name}")

=== QLoRA v2 ===

=== QLoRA v1 (broken) ===
  README.md
  checkpoint-500/
  checkpoint-970/


In [5]:
############################################################
# QLoRA v2 — FULL SELF-CONTAINED CELL
# Install → Load → Train → Save → Test
############################################################
!pip install -q trl peft bitsandbytes accelerate datasets transformers

import torch, gc, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
QLORA_DIR = DRIVE / 'models' / 'qwen-qlora-v2'
QLORA_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM = (
    "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). "
    "İlgili kanun maddelerine atıfta bulun. Bilgi yoksa 'Bu konuda yeterli bilgi bulunamadı' de."
)

# ── Dataset: raw messages only ──
print("Building dataset...")
qa1 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_qa.parquet'))
qa2 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_chatbot.parquet'))
qa1 = qa1.rename(columns={'question': 'query', 'answer': 'answer'})
qa2 = qa2.rename(columns={'Soru': 'query', 'Cevap': 'answer'})
qa_all = pd.concat([qa1[['query','answer']], qa2[['query','answer']]], ignore_index=True).dropna().reset_index(drop=True)
qa_sample = qa_all.sample(min(10000, len(qa_all)), random_state=42).reset_index(drop=True)

training_examples = []
for _, row in qa_sample.iterrows():
    training_examples.append({"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: {row['query']}"},
        {"role": "assistant", "content": row['answer']},
    ]})

unanswerable = ["Bu konuda verilen bağlamda yeterli bilgi bulunmamaktadır.",
    "Bu soruya yanıt verilebilmesi için ek bilgiye ihtiyaç vardır."]
for i in range(300):
    training_examples.append({"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: Hayali soru #{i}"},
        {"role": "assistant", "content": random.choice(unanswerable)},
    ]})
random.shuffle(training_examples)
dataset = Dataset.from_list(training_examples)
print(f"Dataset: {len(dataset)} examples, columns: {dataset.column_names}")
del qa1, qa2, qa_all, qa_sample; gc.collect()

# ── Load model ──
print("Loading Qwen2.5-7B-Instruct...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

# ── Train ──
training_args = SFTConfig(
    output_dir=str(QLORA_DIR / 'checkpoints'),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_steps=30,
    weight_decay=0.01,
    bf16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_length=2048,
)

trainer = SFTTrainer(model=model, args=training_args,
    train_dataset=dataset, processing_class=tokenizer)

steps = len(dataset) // (2 * 16)
print(f"\nTraining: {steps} steps, 1 epoch")
trainer.train()

# ── Save ──
print("\nSaving adapter...")
trainer.model.save_pretrained(str(QLORA_DIR / 'adapter'))
tokenizer.save_pretrained(str(QLORA_DIR / 'adapter'))
print(f"Saved to {QLORA_DIR / 'adapter'}")

# ── Quick test ──
print("\nTesting...")
test_q = "Kasten adam öldürme suçunun cezası nedir?"
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Soru: {test_q}"}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
with torch.inference_mode():
    out = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.1,
        do_sample=True, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {answer[:400]}")
print("\nDONE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.7 MB/s eta 0:00:00
Building dataset...
Dataset: 10300 examples, columns: ['messages']
Loading Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


Training: 321 steps, 1 epoch


Step,Training Loss
25,2.693915
50,1.282157
75,0.935120
100,0.861683
125,0.851588
150,0.815512
175,0.832788
200,0.808785
225,0.792342
250,0.792595



Saving adapter...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Saved to /content/drive/MyDrive/hukuk-rag/models/qwen-qlora-v2/adapter

Testing...
Q: Kasten adam öldürme suçunun cezası nedir?
A: KSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSenSen

DONE.


In [6]:
############################################################
# FIX: Disable gradient checkpointing + enable cache for inference
############################################################
trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True
trainer.model.eval()

# Test again
test_q = "Kasten adam öldürme suçunun cezası nedir?"
msgs = [
    {"role": "system", "content": "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun."},
    {"role": "user", "content": f"Soru: {test_q}"}
]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

with torch.inference_mode():
    out = trainer.model.generate(
        **inputs, max_new_tokens=200, temperature=0.1,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,  # penalize repetition
    )
answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {answer[:400]}")
print(f"Length: {len(answer)} chars")

Q: Kasten adam öldürme suçunun cezası nedir?
A: Kasten adam öldürme, 15 yıl veya daha fazlasına kadar hapis cezasıdır.
Length: 70 chars


In [7]:
# Test a few more questions
test_qs = [
    "Boşanma davasında nafaka nasıl belirlenir?",
    "İdari yargıda dava açma süresi ne kadardır?",
    "Anonim şirketlerde yönetim kurulu üyelerinin sorumluluğu nedir?",
]

for q in test_qs:
    msgs = [
        {"role": "system", "content": "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun."},
        {"role": "user", "content": f"Soru: {q}"}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.inference_mode():
        out = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.2)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {answer[:300]}")

print("\nQLoRA v2 model is WORKING.")


Q: Boşanma davasında nafaka nasıl belirlenir?
A: Boşanma davası açıldığında, mahkeme eşlerin ihtiyaçlarını karşılamak üzere gerekli olan mal varlıklarını değerlendirip, her iki taraf için gereken nafaka miktarını kararlaştırır.

Q: İdari yargıda dava açma süresi ne kadardır?
A: İdari yargılarda, idarenin kararını alması tarihinden itibaren 60 gün içinde davaya başvurulmalıdır.

Q: Anonim şirketlerde yönetim kurulu üyelerinin sorumluluğu nedir?
A: Anonim şirketlerde, yönetim kurulu üyeleri, kurulda karar verilen işlemlerle ilgili olarak sorumlu olurlar; ancak bu tür işlemlerin doğrultusunda yapılan işleyişiyle ilgili olarak da sorumludurlar.

QLoRA v2 model is WORKING.


In [9]:
!pip install -q faiss-cpu rank_bm25 rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 114.0 MB/s eta 0:00:00


In [10]:
############################################################
# CONFIG 4 EVAL: FT Embed + BM25 + RRF + QLoRA LLM
# Load retrieval components (CPU), keep QLoRA on GPU
############################################################
import faiss, pickle, json, time, re
from pathlib import Path
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import pyarrow.parquet as pq

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
TURKISH_LOWER_MAP = str.maketrans("İIÖÜÇŞĞ", "iıöüçşğ")
TURKISH_STOPWORDS = {"bir","bu","da","de","ve","ile","için","olan","olarak","gibi","daha","en","çok","her","kadar","sonra","önce","ise","ya","ne","nasıl","neden","nerede","kim","hangi","o","şu","ben","sen","biz","siz","onlar","mi","mu","mü","mı","dir","dır","dur","dür","tir","tır","tur","tür","ki","ama","ancak","fakat","lakin","veya","yahut","hem","üzere","göre","karşı"}

def normalize_turkish(text):
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def turkish_tokenize(text):
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return [t for t in text.split() if t not in TURKISH_STOPWORDS and len(t) > 1]

# Load FAISS (fine-tuned)
print("Loading FAISS...")
ft_index = faiss.read_index(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.index'))
with open(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.mapping.pkl'), 'rb') as f:
    ft_ids = pickle.load(f)
ft_index.nprobe = 16

# Load BM25
print("Loading BM25...")
with open(str(DRIVE / 'indexes' / 'bm25.pkl'), 'rb') as f:
    bm25_data = pickle.load(f)
bm25_index_obj = bm25_data['index']
bm25_mapping = bm25_data['mapping']

# Load chunk texts
print("Loading chunk texts...")
pf = pq.ParquetFile(str(DRIVE / 'data' / 'processed' / 'chunks_filtered.parquet'))
chunk_text_list = []
for batch in pf.iter_batches(batch_size=100_000, columns=['text']):
    chunk_text_list.extend(batch.column('text').to_pylist())
print(f"  {len(chunk_text_list):,} texts")

# Load embedding model (CPU)
print("Loading embedding model (CPU)...")
embed_model = SentenceTransformer(
    str(DRIVE / 'models' / 'e5-checkpoints' / 'checkpoint-10000'), device='cpu')

# Load gold set
import os
REPO = Path('/content/hukuk-rag')
if not REPO.exists():
    from google.colab import userdata
    try: token = userdata.get("GITHUB_TOKEN")
    except: token = ""
    os.system(f"git clone https://{token}@github.com/berkay-aktas/hukuk-rag.git {REPO}")
with open(REPO / 'data' / 'gold' / 'gold_test_set.json', encoding='utf-8') as f:
    gold_data = json.load(f)['questions']

print(f"Gold set: {len(gold_data)} questions")
print("All loaded.")

Loading FAISS...
Loading BM25...
Loading chunk texts...
  2,496,668 texts
Loading embedding model (CPU)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Gold set: 225 questions
All loaded.


In [11]:
############################################################
# CONFIG 4 RAG PIPELINE + EVAL (225 questions)
############################################################

SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanısın. Soruyu verilen bağlam paragraflarını kullanarak "
    "kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun. "
    "Bağlamda bilgi yoksa 'Bu konuda yeterli bilgi bulunamadı' de."
)

# Use trainer.model (QLoRA) for generation
qlora_model = trainer.model

def dense_search(query, k=50):
    q_emb = embed_model.encode([f"query: {query}"], normalize_embeddings=True).astype(np.float32)
    scores, indices = ft_index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        results.append({"chunk_id": ft_ids[idx], "score": float(score), "text": chunk_text_list[idx]})
    return results

def bm25_search(query, k=50):
    tokens = turkish_tokenize(query)
    scores = bm25_index_obj.get_scores(tokens)
    top_idx = np.argsort(scores)[-k:][::-1]
    results = []
    for idx in top_idx:
        if scores[idx] <= 0: continue
        chunk = bm25_mapping[idx]
        results.append({"chunk_id": chunk["chunk_id"], "score": float(scores[idx]), "text": chunk["text"]})
    return results

def rrf_merge(dense_results, bm25_results, k=60, top_k=10):
    scores = defaultdict(float)
    best = {}
    for results in [dense_results, bm25_results]:
        for rank, r in enumerate(results):
            scores[r["chunk_id"]] += 1.0 / (k + rank + 1)
            if r["chunk_id"] not in best or r["score"] > best[r["chunk_id"]]["score"]:
                best[r["chunk_id"]] = r
    sorted_ids = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)[:top_k]
    return [{"chunk_id": cid, "score": scores[cid], "text": best[cid]["text"]} for cid in sorted_ids]

@torch.inference_mode()
def rag_answer_c4(question):
    dense = dense_search(question, k=50)
    sparse = bm25_search(question, k=50)
    merged = rrf_merge(dense, sparse, k=60, top_k=10)
    context = "\n\n".join(f"[{i+1}] {r['text']}" for i, r in enumerate(merged))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Bağlam:\n{context}\n\nSoru: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(qlora_model.device)
    outputs = qlora_model.generate(**inputs, max_new_tokens=256, temperature=0.1, top_p=0.9,
        do_sample=True, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.2)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), merged

# Run eval
preds_c4 = []
refs_c4 = []
t0 = time.time()

for i, item in enumerate(tqdm(gold_data, desc="Config 4 Eval")):
    try:
        answer, _ = rag_answer_c4(item["question"])
        preds_c4.append(answer)
    except Exception as e:
        preds_c4.append("")
    refs_c4.append(item["gold_answer"])
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(gold_data)} done | {(time.time()-t0)/60:.1f}min")

c4_time = time.time() - t0
print(f"\nDone! {len(preds_c4)} answers in {c4_time/60:.1f} minutes")

Config 4 Eval:   0%|          | 0/225 [00:00<?, ?it/s]

  50/225 done | 20.9min
  100/225 done | 41.1min
  150/225 done | 68.1min
  200/225 done | 88.0min

Done! 225 answers in 99.8 minutes


In [12]:
############################################################
# CONFIG 4 METRICS + FULL COMPARISON (C1 vs C2 vs C3 vs C4)
############################################################
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

# C4 metrics
em_c4 = sum(1 for p, r in zip(preds_c4, refs_c4) if normalize_turkish(p) == normalize_turkish(r)) / len(preds_c4)

f1_c4 = []
for pred, ref in zip(preds_c4, refs_c4):
    pt = Counter(normalize_turkish(pred).split())
    rt = Counter(normalize_turkish(ref).split())
    if not pt or not rt: f1_c4.append(0.0); continue
    common = sum((pt & rt).values())
    if common == 0: f1_c4.append(0.0); continue
    p = common / sum(pt.values())
    r = common / sum(rt.values())
    f1_c4.append(2 * p * r / (p + r))
token_f1_c4 = np.mean(f1_c4)

rouge_c4 = [scorer.score(normalize_turkish(ref), normalize_turkish(pred))["rougeL"].fmeasure
            for pred, ref in zip(preds_c4, refs_c4)]
rouge_l_c4 = np.mean(rouge_c4)

# Load previous results
with open(str(DRIVE / 'results' / 'config1_baseline_225.json')) as f:
    c1 = json.load(f)['metrics']['generation']
with open(str(DRIVE / 'results' / 'config2_finetuned_embeddings.json')) as f:
    c2 = json.load(f)['metrics']['generation']
with open(str(DRIVE / 'results' / 'config3_reranker.json')) as f:
    c3 = json.load(f)['metrics']['generation']

print("=" * 80)
print("  ABLATION RESULTS: C1 → C2 → C3 → C4 (225 questions)")
print("=" * 80)
print(f"{'Metric':<12} {'C1 Baseline':<15} {'C2 +FT Emb':<15} {'C3 +Reranker':<15} {'C4 +QLoRA':<15}")
print("-" * 80)
print(f"{'EM':<12} {c1['exact_match']:<15.4f} {c2['exact_match']:<15.4f} {c3['exact_match']:<15.4f} {em_c4:<15.4f}")
print(f"{'Token F1':<12} {c1['token_f1']:<15.4f} {c2['token_f1']:<15.4f} {c3['token_f1']:<15.4f} {token_f1_c4:<15.4f}")
print(f"{'ROUGE-L':<12} {c1['rouge_l']:<15.4f} {c2['rouge_l']:<15.4f} {c3['rouge_l']:<15.4f} {rouge_l_c4:<15.4f}")
print("-" * 80)
print(f"\n  C1→C2: F1 {((c2['token_f1']-c1['token_f1'])/c1['token_f1']*100):+.1f}%")
print(f"  C2→C3: F1 {((c3['token_f1']-c2['token_f1'])/c2['token_f1']*100):+.1f}%")
print(f"  C3→C4: F1 {((token_f1_c4-c3['token_f1'])/c3['token_f1']*100):+.1f}%")
print(f"  C1→C4: F1 {((token_f1_c4-c1['token_f1'])/c1['token_f1']*100):+.1f}%")

# Save C4 results
config4_results = {
    "config": "Config 4 - FT Embeddings + QLoRA LLM",
    "embedding": "intfloat/multilingual-e5-large (fine-tuned checkpoint-10000)",
    "reranker": "none (skipped — C3 showed regression)",
    "llm": "Qwen/Qwen2.5-7B-Instruct (QLoRA v2, 1 epoch, 10.3k examples)",
    "metrics": {"generation": {"exact_match": float(em_c4), "token_f1": float(token_f1_c4), "rouge_l": float(rouge_l_c4)}},
    "gold_set_size": len(preds_c4),
    "runtime_minutes": round(c4_time / 60, 1),
}
with open(str(DRIVE / 'results' / 'config4_qlora.json'), 'w', encoding='utf-8') as f:
    json.dump(config4_results, f, ensure_ascii=False, indent=2)

# Save predictions
config4_preds = {"config": "Config 4", "predictions": preds_c4, "references": refs_c4,
    "per_question": [{"question": gold_data[i]["question"], "prediction": preds_c4[i],
     "reference": refs_c4[i], "domain": gold_data[i].get("domain",""),
     "difficulty": gold_data[i].get("difficulty",""),
     "is_answerable": gold_data[i].get("is_answerable", True)} for i in range(len(preds_c4))]}
with open(str(DRIVE / 'results' / 'config4_predictions.json'), 'w', encoding='utf-8') as f:
    json.dump(config4_preds, f, ensure_ascii=False, indent=2)

print("\nResults saved.")

  ABLATION RESULTS: C1 → C2 → C3 → C4 (225 questions)
Metric       C1 Baseline     C2 +FT Emb      C3 +Reranker    C4 +QLoRA      
--------------------------------------------------------------------------------
EM           0.0000          0.0000          0.0000          0.0000         
Token F1     0.0950          0.1063          0.0911          0.1509         
ROUGE-L      0.1016          0.1120          0.0978          0.1632         
--------------------------------------------------------------------------------

  C1→C2: F1 +11.9%
  C2→C3: F1 -14.3%
  C3→C4: F1 +65.6%
  C1→C4: F1 +58.8%

Results saved.


In [13]:
  ############################################################
# CONFIG 5: Fully Optimized
# FT Embed + BM25 + RRF + Off-shelf Reranker + QLoRA LLM
############################################################

# Load off-shelf reranker (CPU) — not fine-tuned, just the base Turkish model
print("Loading off-shelf reranker (CPU)...")
from transformers import AutoModelForSequenceClassification
reranker_tok = AutoTokenizer.from_pretrained("seroe/bge-reranker-v2-m3-turkish-triplet")
reranker_mdl = AutoModelForSequenceClassification.from_pretrained(
    "seroe/bge-reranker-v2-m3-turkish-triplet", num_labels=1)
reranker_mdl.eval()
print("  Loaded.")

def rerank(query, passages, top_k=10):
    pairs_scores = []
    for i in range(0, len(passages), 8):
        batch = passages[i:i+8]
        inputs = reranker_tok(
            [query]*len(batch), [p["text"] for p in batch],
            return_tensors='pt', truncation=True, max_length=512, padding=True)
        with torch.inference_mode():
            out = reranker_mdl(**inputs).logits.squeeze(-1)
        if out.dim() == 0:
            pairs_scores.append((batch[0], out.item()))
        else:
            pairs_scores.extend(zip(batch, out.tolist()))
    ranked = sorted(pairs_scores, key=lambda x: x[1], reverse=True)
    return [{"chunk_id": p["chunk_id"], "score": s, "text": p["text"]} for p, s in ranked[:top_k]]

@torch.inference_mode()
def rag_answer_c5(question):
    dense_r = dense_search(question, k=50)
    sparse_r = bm25_search(question, k=50)
    merged = rrf_merge(dense_r, sparse_r, k=60, top_k=30)
    reranked = rerank(question, merged, top_k=10)
    context = "\n\n".join(f"[{i+1}] {r['text']}" for i, r in enumerate(reranked))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Bağlam:\n{context}\n\nSoru: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(qlora_model.device)
    outputs = qlora_model.generate(**inputs, max_new_tokens=256, temperature=0.1, top_p=0.9,
        do_sample=True, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.2)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), reranked

# Quick test
print("\nTesting C5...")
ans, _ = rag_answer_c5("Kasten adam öldürme suçunun cezası nedir?")
print(f"A: {ans[:300]}")
print("C5 pipeline ready.")

Loading off-shelf reranker (CPU)...


config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Loaded.

Testing C5...
A: Kasten adam öldürme suçunun cezası, 5237 sayılı Türk Ceza Kanunu’nun 81. maddesinin birinci fıkrası, 62. maddesinin birinci fıkrası, 58. maddesinin altıncı ve yedinci fıkraları ve 53. üncü maddesinin birinci fıkrası uyarınca 25 yıl hapis cezasıdır.
C5 pipeline ready.


In [14]:
############################################################
# CONFIG 5 EVAL ON 225 QUESTIONS
############################################################
preds_c5 = []
refs_c5 = []
t0 = time.time()

for i, item in enumerate(tqdm(gold_data, desc="Config 5 Eval")):
    try:
        answer, _ = rag_answer_c5(item["question"])
        preds_c5.append(answer)
    except Exception as e:
        preds_c5.append("")
    refs_c5.append(item["gold_answer"])
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(gold_data)} done | {(time.time()-t0)/60:.1f}min")

c5_time = time.time() - t0
print(f"\nDone! {len(preds_c5)} answers in {c5_time/60:.1f} minutes")

# Compute metrics
em_c5 = sum(1 for p, r in zip(preds_c5, refs_c5) if normalize_turkish(p) == normalize_turkish(r)) / len(preds_c5)
f1_c5 = []
for pred, ref in zip(preds_c5, refs_c5):
    pt = Counter(normalize_turkish(pred).split())
    rt = Counter(normalize_turkish(ref).split())
    if not pt or not rt: f1_c5.append(0.0); continue
    common = sum((pt & rt).values())
    if common == 0: f1_c5.append(0.0); continue
    p = common / sum(pt.values())
    r = common / sum(rt.values())
    f1_c5.append(2 * p * r / (p + r))
token_f1_c5 = np.mean(f1_c5)
rouge_c5 = [scorer.score(normalize_turkish(ref), normalize_turkish(pred))["rougeL"].fmeasure
            for pred, ref in zip(preds_c5, refs_c5)]
rouge_l_c5 = np.mean(rouge_c5)

# Full comparison
print("=" * 90)
print("  FINAL ABLATION: C1 → C2 → C3 → C4 → C5 (225 questions)")
print("=" * 90)
print(f"{'Metric':<12} {'C1 Base':<12} {'C2 +Emb':<12} {'C3 +Rerank':<12} {'C4 +QLoRA':<12} {'C5 Full':<12}")
print("-" * 90)
print(f"{'EM':<12} {c1['exact_match']:<12.4f} {c2['exact_match']:<12.4f} {c3['exact_match']:<12.4f} {em_c4:<12.4f} {em_c5:<12.4f}")
print(f"{'Token F1':<12} {c1['token_f1']:<12.4f} {c2['token_f1']:<12.4f} {c3['token_f1']:<12.4f} {token_f1_c4:<12.4f} {token_f1_c5:<12.4f}")
print(f"{'ROUGE-L':<12} {c1['rouge_l']:<12.4f} {c2['rouge_l']:<12.4f} {c3['rouge_l']:<12.4f} {rouge_l_c4:<12.4f} {rouge_l_c5:<12.4f}")
print("-" * 90)
print(f"\n  C1→C5 overall: F1 {((token_f1_c5-c1['token_f1'])/c1['token_f1']*100):+.1f}%, ROUGE-L {((rouge_l_c5-c1['rouge_l'])/c1['rouge_l']*100):+.1f}%")

# Save
config5 = {
    "config": "Config 5 - Fully Optimized",
    "embedding": "intfloat/multilingual-e5-large (fine-tuned)",
    "reranker": "seroe/bge-reranker-v2-m3-turkish-triplet (off-shelf)",
    "llm": "Qwen/Qwen2.5-7B-Instruct (QLoRA v2)",
    "metrics": {"generation": {"exact_match": float(em_c5), "token_f1": float(token_f1_c5), "rouge_l": float(rouge_l_c5)}},
    "gold_set_size": len(preds_c5), "runtime_minutes": round(c5_time/60, 1),
}
with open(str(DRIVE / 'results' / 'config5_full.json'), 'w', encoding='utf-8') as f:
    json.dump(config5, f, ensure_ascii=False, indent=2)
c5_preds = {"config": "Config 5", "predictions": preds_c5, "references": refs_c5,
    "per_question": [{"question": gold_data[i]["question"], "prediction": preds_c5[i],
     "reference": refs_c5[i], "domain": gold_data[i].get("domain",""),
     "difficulty": gold_data[i].get("difficulty",""),
     "is_answerable": gold_data[i].get("is_answerable", True)} for i in range(len(preds_c5))]}
with open(str(DRIVE / 'results' / 'config5_predictions.json'), 'w', encoding='utf-8') as f:
    json.dump(c5_preds, f, ensure_ascii=False, indent=2)
print("\nAll results saved.")

Config 5 Eval:   0%|          | 0/225 [00:00<?, ?it/s]

  50/225 done | 36.8min
  100/225 done | 79.0min
  150/225 done | 121.3min
  200/225 done | 159.7min

Done! 225 answers in 181.1 minutes
  FINAL ABLATION: C1 → C2 → C3 → C4 → C5 (225 questions)
Metric       C1 Base      C2 +Emb      C3 +Rerank   C4 +QLoRA    C5 Full     
------------------------------------------------------------------------------------------
EM           0.0000       0.0000       0.0000       0.0000       0.0000      
Token F1     0.0950       0.1063       0.0911       0.1509       0.1431      
ROUGE-L      0.1016       0.1120       0.0978       0.1632       0.1530      
------------------------------------------------------------------------------------------

  C1→C5 overall: F1 +50.6%, ROUGE-L +50.6%

All results saved.
